# 추론 모델 밑바닥부터 7편 실습 노트북 — 8장: 추론 모델 증류하기 (한국어 주석판)

**출처:** 원본 노트북 `ch08_main.ipynb`, Sebastian Raschka, https://github.com/rasbt/reasoning-from-scratch , Apache License 2.0. 책 *Build a Reasoning Model (From Scratch)*, Manning.

**변경 사항:** 코난쌤(@conanssam)이 설명 글을 한국어로 옮기고 코드에 한국어 주석을 덧붙였습니다. 코드 동작은 원본과 같습니다.

**쓰는 법**
1. 저장소를 내려받습니다: `git clone https://github.com/rasbt/reasoning-from-scratch.git`
2. 저장소 폴더에서 `uv sync`를 실행해 필요한 패키지를 설치합니다.
3. `uv run jupyter lab`으로 주피터랩을 열고, 이 파일을 `ch08/01_main-chapter-code/` 폴더에 넣어 실행합니다.

**팁:** 증류 학습은 GPU가 있어야 현실적입니다. 처음에는 데이터와 스텝을 작게 잡아 돌려 보세요.

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a>의 책 <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a>에 딸린 실습 코드입니다<br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp?1" width="100px"></a>
</td>
</tr>
</table>

# 8장: 효율적인 추론을 위해 추론 모델 증류하기

이 노트북에서 쓰는 패키지입니다.

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

# 설치된 패키지 버전을 하나씩 출력합니다
for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F01_raschka.webp" width=600>

&nbsp;
## 8.1 추론 작업을 위한 모델 증류 소개

- 증류(distillation)란 작은 LLM(“학생”)을 큰 LLM(“선생”)이 낸 출력으로 학습시키는 것입니다.
- 증류는 RLVR로 처음부터 학습하는 것보다 효과가 좋을 수 있습니다.
  - 예를 들어 [DeepSeek-R1 논문](https://arxiv.org/abs/2501.12948)을 보면, 작은 DeepSeek-R1 모델들은 RLVR보다 증류로 더 좋은 성능을 냈다고 합니다.
  - 구체적으로는 가장 큰 DeepSeek R1 모델(파라미터 6,710억 개)을 먼저 학습했습니다. 그다음 이 모델로 작은 모델들을 증류해 학습시켰습니다.

- 증류에는 크게 두 가지가 있습니다. 하드 증류(hard distillation)와 소프트 증류(soft distillation)입니다.
  1. 하드 증류: 학생을 선생이 만든 글로 학습시킵니다. 선생의 출력을 정답 목표로 여깁니다.
  2. 소프트 증류: 학생이 어휘 전체에 대한 선생의 확률 분포를 따라 하도록 학습시킵니다. 보통 두 모델의 로짓(logit)에서 얻은 분포 사이의 KL 발산(KL divergence)을 줄이는 방식입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F02_raschka.webp" width=600>

- 1. 순수 하드 증류:
  - 선생 모델(teacher model)이 만든 글만 목표로 씁니다. 여기서는 선생의 출력을 정답 목표로 봅니다.
  - 손실: `L = CrossEntropy(y_teacher_tokens, y_student)`
  - 엄밀히 말하면 합성 데이터로 하는 일반적인 지도 미세 조정(supervised fine-tuning)입니다.
  - 선생의 로짓이 없어도 됩니다.
  - DeepSeek-R1의 작은 모델들이 바로 이렇게 증류했습니다. 6,710억 파라미터의 큰 DeepSeek-R1(선생)로 추론 과정을 만들고, 작은 LLM(학생 모델, student model)을 그것으로 미세 조정했습니다.

- 2. 순수 소프트 증류
  - 선생의 출력 확률 분포를 그대로 따라 하게 합니다.
  - 손실: `L = KL(p_teacher || p_student)`
  - 학습할 때 선생의 로짓(또는 로그 확률)이 있어야 합니다.

- 3. 하드 증류와 소프트 증류 합치기
  - 고전적인 지식 증류(knowledge distillation)입니다. 컴퓨터 비전에서 시작했고 [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531)에 나옵니다.
  - 선생이 실제로 낸 출력 토큰으로 학습합니다. 동시에 학생이 선생의 전체 확률 분포도 따라 하게 만듭니다.
  - 손실: `L = CE(y_teacher_tokens, y_student) + lambda * KL(p_teacher || p_student)`. 여기서 `lambda`는 KL 항이 얼마나 반영될지 정하는 하이퍼파라미터입니다.

- 실제로는 하드 증류를 더 많이 씁니다.
  - 로짓은 보통 얻을 수 없습니다. 예를 들어 OpenAI ChatGPT나 Anthropic Claude 같은 상용 시스템은 어휘 전체의 로짓을 공개하지 않습니다. 그래서 고전적인 소프트 증류는 할 수 없습니다.
  - 로짓을 얻을 수 있어도, 소프트 증류를 하려면 학생과 선생 LLM이 같은 토큰화 방식을 써야 합니다. 그렇지 않으면 어휘가 맞지 않습니다. 그래서 소프트 증류는 같은 모델 계열 안에서 더 쓸모가 있습니다.
  - 글 출력을 만들어 증류에 다시 쓰는 것만으로도 일부 제공업체의 정책을 어길 수 있다는 점을 알아 두세요.
  - 로짓이 있어도 긴 시퀀스의 어휘 전체 분포를 저장하려면 대역폭과 디스크 공간이 많이 듭니다. 일반 글 출력을 만들어 저장하는 쪽이 훨씬 쌉니다.

- 소프트 증류와 하드 증류를 둘 다 할 수 있다면, 어느 쪽이 더 좋을까요?
  - 목표 함수를 잘 설계하면 소프트 증류가 하드 증류보다 나을 수 있습니다(https://arxiv.org/abs/2306.08543).
  - 하지만 최신 LLM 벤치마크에서 소프트냐 하드냐만 바꿔 공정하게 비교한 최근의 대규모 실험은 없습니다.
  - 감독 방식이 소프트냐 하드냐보다 데이터를 어떻게 만드느냐가 더 중요할 수 있습니다(https://arxiv.org/abs/2306.13649).

- 이 장에서는 DeepSeek-R1과 비슷한 하드 증류를 구현합니다. 더 흔하고 실용적인 방법입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F03_raschka.webp" width=600>

&nbsp;
## 8.2 추론 증류용 데이터셋 만들기

- 증류의 첫 단계는 학생 모델을 학습시킬 데이터셋을 만드는 것입니다. 여기서 학생은 이 책 내내 다룬 Qwen3 0.6B 기본(base) 모델입니다.
- MATH 데이터셋에서 MATH-500 테스트 세트와 겹치지 않는 수학 문제 12,000개를 씁니다(https://github.com/rasbt/math_full_minus_math500).
- 6장과 7장에서 RLVR에 썼던 바로 그 12,000문제입니다.
- 이 12,000문제를 이미 있는 추론 모델(여기서는 DeepSeek-R1)에 넣고, 나온 답을 모아 학습에 씁니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F04_raschka.webp" width=500>

- 6장과 7장에서는 Qwen3 기본 모델이 맞는 풀이를 내도록 RLVR로 학습시켰습니다.
- 그때는 모델이 낸 풀이를 검증기(verifier)로 참고 풀이와 비교해 학습 신호를 계산했습니다.
- 증류에서는 선생의 풀이와 학생의 풀이를 비교해 학습 신호를 계산합니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F05_raschka.webp" width=600>

- 이번 증류 방식에서는 학생 모델 학습을 시작하기 전에, 선생 모델로 증류 데이터셋을 미리 만들어 둘 수 있습니다.
- 12,000문제 전부에 선생의 답을 만들려면 시간과 자원이 많이 듭니다. 그래서 저자가 OpenRouter로 클라우드에 올라간 6,710억 파라미터 DeepSeek-R1 모델을 써서 데이터셋을 미리 만들어 두었습니다(API 비용으로 약 50달러가 들었습니다).
- 다음 절에서 미리 만든 이 데이터셋을 불러옵니다. 그러니 여러분이 직접 만들 필요는 없습니다.
- 그래도 궁금하다면, 저자가 쓴 코드와 사용법을 부록 자료 [https://github.com/rasbt/reasoning-from-scratch/tree/main/ch08/02_generate_distillation_data](https://github.com/rasbt/reasoning-from-scratch/tree/main/ch08/02_generate_distillation_data)에서 볼 수 있습니다.

&nbsp;
## 8.3 증류용 MATH 학습 데이터셋 불러오기

- 이 절에서는 앞 절에서 설명한 대로 DeepSeek-R1이 만든 수학 문제와 답 데이터셋을 불러옵니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F06_raschka.webp" width=600>

- 데이터셋은 Hugging Face 허브 https://huggingface.co/datasets/rasbt/math_distill 에 있습니다.
- 아래 코드로 파이썬 세션에 불러옵니다.

In [ ]:
import json
import requests
from pathlib import Path


def load_distill_data(
    local_path=None,
    partition="deepseek-r1-math-train",
    save_copy=True,
):

    if local_path is None:
        local_path = f"{partition}.json"
    local_path = Path(local_path)

    url = (
        "https://huggingface.co/datasets/rasbt/math_distill"
        "/resolve/main/data/"
        f"{partition}.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        f"MATH/{partition}.json"
    )

    # 이미 받아 둔 파일이 있으면 그것을 씁니다
    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        size_kb = local_path.stat().st_size / 1e3
        print(f"{local_path}: {size_kb:.1f} KB (cached)")
        return data

    assert partition in (
        "deepseek-r1-math-train",
        "deepseek-r1-math500",
        "qwen3-235b-a22b-math-train",
        "qwen3-235b-a22b-math500",
    )

    # 먼저 Hugging Face에서 받고, 실패하면 예비 주소를 씁니다
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
    except requests.RequestException:
        print("Using backup URL.")
        r = requests.get(backup_url, timeout=30)
        r.raise_for_status()

    data = r.json()

    # 다음에 다시 쓰도록 내려받은 데이터를 저장합니다
    if save_copy:
        with local_path.open("w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

        size_kb = local_path.stat().st_size / 1e3
        print(f"{local_path}: {size_kb:.1f} KB")

    return data

In [ ]:
# DeepSeek-R1이 만든 학습용 증류 데이터를 불러옵니다
math_train = load_distill_data(partition="deepseek-r1-math-train")

print("Dataset size:", len(math_train))

- 학습 예제 하나를 열어 구조를 확인해 봅시다.

In [ ]:
from pprint import pprint

# 다섯 번째 예제를 보기 좋게 출력합니다
pprint(math_train[4])

In [ ]:
math_train[4].keys()

- 데이터셋에는 네 가지 필드가 있습니다.
  - `gtruth_answer`: 맞는 참고 답(정답, ground truth answer)
  - `message_thinking`: 모델이 쓴 추론 과정(reasoning trace)
  - `message_content`: 추론 과정 뒤에 나오는 최종 답
- 아래 코드로 답을 형식에 맞게 정리할 수 있습니다.

In [ ]:
def format_distilled_answer(entry):
    content = str(entry["message_content"]).strip()
    if not content:
        raise ValueError("Missing non-empty 'message_content' field.")

    # 추론 과정을 <think> 태그로 감싸고 최종 답을 뒤에 붙입니다
    thinking = str(entry["message_thinking"]).strip()
    return f"<think>{thinking}</think>\n\n{content}"

In [ ]:
print(format_distilled_answer(math_train[4]))

- 앞 장에서 말했듯 `<think></think>` 토큰이 꼭 필요한 것은 아닙니다.
- 다만 추론 과정과 최종 답을 나누는 데 쓸모가 있습니다.
- OpenAI의 ChatGPT처럼 추론 과정을 사용자에게 숨기는 제공업체나 화면도 있습니다. 이때 `<think></think>` 토큰을 쓰고 모델이 이 토큰을 쓰도록 가르쳐 두면, 생각 과정을 골라내 숨기기가 쉬워집니다.

- DeepSeek-R1 모델이 이 데이터셋에서 정확도가 얼마나 나왔는지 궁금할 수도 있습니다. 데이터셋에 정답 레이블이 있으니 3장의 검증기로 정확도를 계산할 수 있습니다.
- 편하게 하려면 부록 자료(3장)의 [evaluate_json.py](https://github.com/rasbt/reasoning-from-scratch/blob/main/ch03/02_math500-verifier-scripts/evaluate_json.py) 스크립트를 쓰면 됩니다. JSON 파일에서 바로 정확도를 계산해 줍니다.

In [ ]:
from reasoning_from_scratch.ch07 import download_from_github

# 3장의 정확도 계산 스크립트를 내려받습니다
_ = download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_json.py"
)

In [ ]:
# 선생 답(message_content)을 정답과 비교해 정확도를 냅니다
!uv run evaluate_json.py \
--json_path "deepseek-r1-math-train.json" \
--gtruth_answer gtruth_answer \
--generated_text message_content

&nbsp;
## 8.4 학습 예제 만들기

- 이 절에서는 학습에 쓸 데이터셋 예제를 준비합니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F07_raschka.webp" width=600>

- 준비는 주로 예제의 형식을 맞추고 토큰으로 나누는 일입니다.
- RLVR에서는 학습 예제를 한 번씩만 썼기 때문에 이 일을 그때그때 했습니다.
- 증류는 지도 미세 조정의 한 형태입니다. 같은 예제를 여러 번 보면 도움이 됩니다.
- 그래서 같은 일을 되풀이하지 않도록, 예제를 한 번만 형식화하고 토큰화해서 저장해 두고 다시 씁니다.

&nbsp;
### 8.4.1 토크나이저 불러오고 이해하기

- 토크나이저(tokenizer)부터 시작합니다.
- 7장에서 다뤘듯 `<think>...</think>` 토큰을 지원하는 "reasoning"(추론) 버전을 씁니다.

In [ ]:
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer
)

def load_reasoning_tokenizer(local_dir="qwen3"):
    # 추론용 토크나이저 파일만 내려받습니다
    download_qwen3_small(
        kind="reasoning", tokenizer_only=True, out_dir=local_dir
    )

    tokenizer_path = Path(local_dir) / "tokenizer-reasoning.json"
    # 채팅 템플릿과 생각(think) 표시를 켠 토크나이저를 만듭니다
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )
    
    return tokenizer


tokenizer = load_reasoning_tokenizer()

- `apply_chat_template=True`와 `add_generation_prompt=True`로 설정했다는 점을 눈여겨보세요. 이렇게 하면 프롬프트에 형식이 더 붙습니다. 아래에서 확인해 봅시다.

In [ ]:
prompt = "Sam is hired for a 20-day period..."

# 인코딩했다가 다시 풀어 보면 템플릿이 붙은 모습이 보입니다
prompt_ids = tokenizer.encode(prompt)
decoded_prompt = tokenizer.decode(prompt_ids)
print(decoded_prompt)

- 여기서 `<|im_start|>user`는 사용자가 준 프롬프트의 시작 토큰입니다.
- `<|im_end|>`는 사용자 프롬프트의 끝(stop) 토큰입니다.
- `<|im_start|>assistant`는 LLM 답의 시작 토큰입니다.
- 이 형식은 꼭 써야 하는 것은 아닙니다. 하지만 지시·채팅 미세 조정에서 흔히 쓰는 유용한 관례입니다.
- 또 답에는 `chat_wrapped=False`로 이 감싸기를 꺼야 합니다. 학습할 때는 프롬프트 토큰과 답 토큰을 이어 붙이므로, 끄지 않으면 시작 토큰이 너무 많아집니다.

In [ ]:
answer = (
    "<think>Okay, let me try to solve "
    "this problem...</think> \\boxed{4}"
)
# 답에는 채팅 감싸기를 끕니다
answer_ids = tokenizer.encode(answer, chat_wrapped=False)
decoded_answer = tokenizer.decode(answer_ids)
print(decoded_answer)

- 학습에는 답 끝에 시퀀스 끝(end-of-sequence) 토큰 `<|im_end|>`까지 붙인 토큰 ID 전체가 필요합니다.

In [ ]:
# 프롬프트 + 답 + 끝 토큰을 이어 붙입니다
token_ids = prompt_ids + answer_ids + [tokenizer.eos_token_id]
decoded_token_ids = tokenizer.decode(token_ids)
print(decoded_token_ids)

- 다시 강조하면, <think></think> 토큰 때문에 추론용 토크나이저를 쓰지만 크게 중요하지는 않습니다. 기본 토크나이저를 써도 됩니다.
- 채팅 템플릿도 선택 사항입니다. Qwen3의 추론 모델이 이 형식을 쓰기 때문에 그대로 둘 뿐입니다(에이전트나 여러 프롬프트를 다룰 때만 차이가 납니다).
- 그래서 평가 스크립트에서도 일관되게 `--which_model "reasoning"`을 써야 한다는 점을 기억하세요.

&nbsp;
### 8.4.2 데이터셋 형식화와 토큰화

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F08_raschka.webp" width=400>

- 아래에서 데이터셋 전체에 필요한 형식을 입히는 `build_examples` 함수를 만듭니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F09_raschka.webp" width=600>

In [ ]:
from reasoning_from_scratch.ch03 import render_prompt

def build_examples(data, tokenizer):
    examples = []
    skipped = 0

    for entry in data:
        try:
            # Step 1: encode prompt
            prompt = render_prompt(entry["problem"])
            prompt_ids = tokenizer.encode(prompt)

            # Step 2: encode answer
            target_answer = format_distilled_answer(entry)
            answer_ids = tokenizer.encode(
                target_answer, chat_wrapped=False
            )

            # Step 3: Combine prompt and answer
            token_ids = (
                prompt_ids + answer_ids + [tokenizer.eos_token_id]
            )

            # 너무 짧은 예제는 건너뜁니다
            if len(token_ids) < 2:
                skipped += 1
                continue

            # 토큰 ID와 프롬프트 길이를 함께 저장합니다
            examples.append({
                "token_ids": token_ids,
                "prompt_len": len(prompt_ids),
            })
        # 필드가 없거나 잘못된 예제도 건너뛴 것으로 셉니다
        except (KeyError, TypeError, ValueError):
            skipped += 1

    return examples, skipped

In [ ]:
# 전체 데이터를 학습 예제로 바꿉니다
examples, skipped = build_examples(math_train, tokenizer)

print("Number of examples:", len(examples))
print("Number of skipped examples:", skipped)

In [ ]:
print(tokenizer.decode(examples[4]["token_ids"]))

&nbsp;
### 8.4.3 데이터셋 거르고 나누기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F10_raschka.webp" width=400>

In [ ]:
def compute_length(examples, answer_only=False):
    lengths = []
    for ex in examples:
        total = len(ex["token_ids"])
        # answer_only=True면 답 부분 길이만 셉니다
        length = total - ex["prompt_len"] if answer_only else total
        lengths.append(length)

    # 평균·최단·최장 길이를 구합니다
    avg_len = round(sum(lengths) / len(lengths))

    shortest_len = min(lengths)
    longest_len = max(lengths)
    shortest_idx = lengths.index(shortest_len)
    longest_idx = lengths.index(longest_len)

    print(f"Average: {avg_len} tokens")
    print(f"Shortest: {shortest_len} tokens (index {shortest_idx})")
    print(f"Longest: {longest_len} tokens (index {longest_idx})")

compute_length(examples)

In [ ]:
def filter_examples_by_max_len(examples, max_len=2048):
    # max_len 토큰보다 긴 예제는 버립니다
    filtered_examples = [
        s for s in examples
        if len(s["token_ids"]) <= max_len
    ]

    print("Original:", len(examples))
    print("Filtered:", len(filtered_examples))
    print("Removed:", len(examples) - len(filtered_examples))

    return filtered_examples

filtered_examples = filter_examples_by_max_len(examples, max_len=2048)

In [ ]:
compute_length(filtered_examples)

In [ ]:
import random

# 시드를 고정해 섞고, 앞 25개를 검증 세트로 뗍니다
rng = random.Random(123)
rng.shuffle(filtered_examples)

train_examples = filtered_examples[25:]
val_examples = filtered_examples[:25]

In [ ]:
print("Number of train examples:", len(train_examples))
print("Number of validation examples:", len(val_examples))

- 연습 문제: 학습 세트와 검증 세트의 길이를 살펴보고 고르게 나뉘었는지 확인해 보세요.

In [ ]:
compute_length(train_examples)

In [ ]:
compute_length(val_examples)

In [ ]:
import matplotlib.pyplot as plt

# 세트마다 예제 길이(토큰 수)를 모읍니다
train_lengths = [len(ex["token_ids"]) for ex in train_examples]
val_lengths = [len(ex["token_ids"]) for ex in val_examples]

# Normalize counts because the validation split is much smaller
bins = range(0, max(train_lengths + val_lengths) + 64, 64)

fig, ax = plt.subplots(figsize=(7, 4))
# 두 세트의 길이 분포를 겹쳐 그립니다
ax.hist(train_lengths, bins=bins, density=True, alpha=0.6, label="Train")
ax.hist(val_lengths, bins=bins, density=True, alpha=0.6, label="Validation")
ax.set_xlabel("Token length")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()


&nbsp;
## 8.5 사전학습된 모델 불러오기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F11_raschka.webp" width=600>

In [ ]:
import torch

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

# 쓸 수 있는 장치(GPU/CPU)를 고릅니다
device = get_device()

# 학생이 될 Qwen3 0.6B 기본(base) 모델을 불러옵니다
model, _ = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

&nbsp;
## 8.6 학습 손실과 검증 손실 계산하기

- 이 절에서는 이른바 교차 엔트로피(cross-entropy) 손실을 계산하는 코드를 구현합니다. 증류할 때 이것을 학습 신호로 씁니다.
- 학습 중 평가를 위해 검증 세트에도 적용합니다.
- 딥러닝을 공부해 봤다면 분류 문제에서 교차 엔트로피를 이미 알고 있을 것입니다. 여기서도 같은 교차 엔트로피 함수입니다. 다만 예측할 목표가 시퀀스의 다음 토큰입니다.
- 교차 엔트로피 손실은 5장과 6장에서 계산한 로그 확률(logprob) 계산에서 끌어낼 수 있습니다.

- 로그 확률 계산을 처음 소개한 5장에서 설명했듯, 로그 확률은 다음에 만들어질 토큰의 점수를 계산합니다.
- 로그 확률이 높으면 모델이 맞는 토큰에 높은 확률을 줬다는 뜻입니다(예측이 더 좋음). 낮으면 낮은 확률을 줬다는 뜻입니다(예측이 더 나쁨).
- 아래 그림은 5장의 예시를 다시 씁니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F12_raschka.webp" width=600>

- 위 그림처럼, 6장의 `sequence_logprob` 함수는 토큰마다 로그 확률(logprob)을 계산한 뒤 합계로 돌려줍니다(곧 결합 로그 확률입니다).

- 학습 세트의 예제 하나로 살펴봅시다. 계산 시간을 줄이려고 가장 짧은 예제를 고릅니다.

In [ ]:
# 5730 is the shortest example in train_examples
token_ids = train_examples[5730]["token_ids"]
prompt_len = train_examples[5730]["prompt_len"]

- `sequence_logprob`가 돌려주는 로그 확률의 합을 그대로 보여 주지 않습니다. 대신 답 토큰 수로 나눠 평균을 냅니다. 답 토큰 수는 프롬프트와 답 토큰을 모두 포함한 전체 토큰 수에서 `prompt_len`을 빼서 구합니다.

In [ ]:
from reasoning_from_scratch.ch06 import sequence_logprob

# 토큰 ID를 텐서로 바꿉니다
tok = torch.tensor(token_ids, dtype=torch.long, device=device)

with torch.no_grad():
    seq_logprob = sequence_logprob(model, tok, prompt_len)
    # 답 토큰 수로 나눠 평균 음의 로그 확률을 구합니다
    num_answer_tokens = tok.numel() - prompt_len
    avg_logprob = -seq_logprob / num_answer_tokens
print(f"Average logprob: {avg_logprob:.2f}")

- 같은 값을 `cross_entropy` 함수로도 계산할 수 있습니다.
- 이 함수는 분류용으로 만들어졌습니다. 그래서 모델의 로짓과 함께 클래스 레이블을 인자로 받습니다. 여기서 클래스 레이블은 “맞는” 토큰입니다.
- 정확히는 답 토큰에 대해서만 교차 엔트로피를 계산합니다(프롬프트 토큰은 무시합니다).

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F13_raschka.webp" width=600>

In [ ]:
# 마지막 토큰을 뺀 것이 입력, 하나씩 민 것이 목표입니다
input_ids = tok[:-1].unsqueeze(0)
target_ids = tok[1:]  # Targets are inputs shifted by one position
logits = model(input_ids).squeeze(0)

# Remove prompt tokens
first_answer_logit_idx = max(prompt_len - 1, 0)
answer_logits = logits[first_answer_logit_idx:]
answer_targets = target_ids[first_answer_logit_idx:]

with torch.no_grad():
    # 답 부분만 교차 엔트로피 평균을 계산합니다
    ce_mean_direct = torch.nn.functional.cross_entropy(
        answer_logits, answer_targets
    )
print(f"Cross-entropy: {ce_mean_direct:.2f}")

- 보다시피 앞에서 답 토큰의 평균 로그 확률을 계산했을 때와 결과가 같습니다.
- 다시 말해 `cross_entropy` 함수는 안에서 평균 로그 확률 계산을 하고 있습니다.
- 직접 만든 `sequence_logprob` 함수 대신 `cross_entropy` 함수를 쓰는 이유는, 더 잘 최적화되어 있어 학습이 빠르기 때문입니다.
- 편하게 부를 수 있도록 함수로 묶어 봅시다.

In [ ]:
def compute_example_loss(model, example, device):
    token_ids = example["token_ids"]
    prompt_len = example["prompt_len"]

    # 입력과 목표를 한 칸 어긋나게 만듭니다
    input_ids = torch.tensor(
        token_ids[:-1], dtype=torch.long, device=device
    ).unsqueeze(0)
    target_ids = torch.tensor(
        token_ids[1:], dtype=torch.long, device=device
    )

    logits = model(input_ids).squeeze(0)

    # 프롬프트 부분은 빼고 답 부분만 남깁니다
    answer_start = max(prompt_len - 1, 0)
    answer_logits = logits[answer_start:]
    answer_targets = target_ids[answer_start:]

    # 답 토큰에 대한 교차 엔트로피가 이 예제의 손실입니다
    loss = torch.nn.functional.cross_entropy(
        answer_logits, answer_targets
    )
    return loss

In [ ]:
with torch.no_grad():
    # 가장 짧은 예제 하나의 손실을 계산합니다
    loss = compute_example_loss(
        model, train_examples[5730], device
    )

print(f"Loss: {loss:.2f}")

- 여러 샘플을 한꺼번에 계산할 수도 있습니다. 이것을 배치 처리(batching)라고 합니다. 하지만 자원 요구량을 낮추려고 여기서는 생략합니다.
- 배치 처리는 부록 E에서 다룹니다.

- 여러 예제에 대해 손실을 계산하는 감싸기(wrapper) 함수도 만들 수 있습니다.

In [ ]:
@torch.no_grad()
def evaluate_examples(model, examples, device):
    was_training = model.training
    # 평가 모드로 바꾸고, 끝나면 원래 모드로 되돌립니다
    model.eval()
    total_loss = 0.0
    num_examples = 0

    # 예제마다 손실을 더해 평균을 냅니다
    for example in examples:
        loss = compute_example_loss(model, example, device)
        total_loss += loss.item()
        num_examples += 1

    if was_training:
        model.train()

    return total_loss / num_examples

In [ ]:
# 학습 세트 앞 3개로 손실을 빠르게 확인합니다
train_loss = evaluate_examples(model, train_examples[:3], device)
print(f"Train loss (3 examples): {train_loss:.2f}")

In [ ]:
# 검증 세트 앞 3개로 손실을 확인합니다
val_loss = evaluate_examples(model, val_examples[:3], device)
print(f"Validation loss (3 examples): {val_loss:.2f}")

- 이 함수는 학습 중 진행 상황을 지켜볼 때 다시 씁니다. 손실이 줄어드는지 보고 싶기 때문입니다.

&nbsp;
## 8.7 증류용 학습 루프 구현하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F14_raschka.webp" width=600>

- 학습 루프는 6장과 거의 같습니다. 다른 점은 학습 세트를 여러 번(에포크, epoch) 돈다는 것, 그리고 GRPO 손실 대신 교차 엔트로피 손실을 계산한다는 것입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F15_raschka.webp" width=500>

In [ ]:
import time


def train_distillation(
    model,
    train_examples,
    val_examples,
    device,
    epochs=2,
    lr=5e-6,
    grad_clip_norm=None,
    seed=123,
    log_every=50,
    checkpoint_dir="checkpoints",
    csv_log_path=None,
):
    # Step 1: initialize optimizer (model is already loaded)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()

    # 전체 스텝 수 = 에포크 수 × 학습 예제 수
    total_steps = epochs * len(train_examples)
    global_step = 0
    rng = random.Random(seed)

    # 로그 파일 이름을 따로 안 주면 시각을 붙여 만듭니다
    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_distill_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    # Step 2: iterate over training epochs
    for epoch in range(1, epochs + 1):
        # Step 3: shuffle the training examples at the start of the epoch
        epoch_examples = list(train_examples)
        rng.shuffle(epoch_examples)

        # Step 4: iterate over training examples in epoch
        for example in epoch_examples:
            global_step += 1

            # Stage 5: reset loss gradient
            # (it's best practice to do this at the beginning of each step)
            optimizer.zero_grad()

            # Step 6: compute the cross-entropy loss for the current example
            loss = compute_example_loss(model, example, device)

            # Step 7: backpropagate gradients
            loss.backward()

            # Optionally clip large gradients to improve training stability
            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), grad_clip_norm
                )

            # Step 8: update the model weights
            optimizer.step()

            # Step 9: periodically evaluate the current model on the validation set
            if log_every and global_step % log_every == 0:
                val_loss = evaluate_examples(
                    model=model,
                    examples=val_examples,
                    device=device,
                )
                model.train()
                # 에포크·스텝·학습 손실·검증 손실을 출력합니다
                print(
                    f"[Epoch {epoch}/{epochs} "
                    f"Step {global_step}/{total_steps}] "
                    f"train_loss={loss.item():.4f} "
                    f"val_loss={val_loss:.4f}"
                )
                append_csv_metrics(
                    csv_log_path=csv_log_path,
                    epoch_idx=epoch,
                    total_steps=global_step,
                    train_loss=loss.item(),
                    val_loss=val_loss,
                )

        # Step 10: save a checkpoint for this epoch
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=global_step,
            suffix=f"epoch{epoch}",
        )
        print(f"Saved checkpoint to {ckpt_path}")
    return model


def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-distill-step{step:05d}{suffix}.pth"
    )
    # 모델 가중치를 .pth 파일로 저장합니다
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    epoch_idx,
    total_steps,
    train_loss,
    val_loss,
):
    # 파일이 없을 때만 머리줄을 씁니다
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "epoch,total_steps,train_loss,val_loss\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{epoch_idx},{total_steps},{train_loss:.6f},"
            f"{val_loss:.6f}\n"
        )

- 시퀀스 길이가 2048이면 학습에 약 15GB의 RAM이 필요합니다.
- 자원 요구량을 낮추려면 긴 시퀀스를 걸러 내면 됩니다. 예를 들어 앞의 `previous filtered_examples = filter_examples_by_max_len(examples, max_len=2048)` 단계에서 `max_len`을 `2048`에서 `1024`나 `512`로 바꿉니다.

In [ ]:
# 결과를 다시 만들 수 있게 시드를 고정합니다
torch.manual_seed(0)

# 보여 주기용으로 예제 10개만 2에포크 학습합니다
train_distillation(
    model,
    train_examples=train_examples[:10],
    val_examples=val_examples[:10],
    device=device,
    epochs=2,
    lr=5e-6,
    grad_clip_norm=1.0,  # Same as in chapter 6
    seed=123,
    log_every=5,
    csv_log_path="train_distill_metrics.csv"
)

- 위 실행은 보여 주기용이라 짧습니다.
- 학습 예제 10개(`train_examples[:10]`)로만 학습하고, 검증 예제 10개(`val_examples`)로만 평가합니다.
- 실제 학습에서는 `train_examples`와 `val_examples`로 바꿉니다.

&nbsp;
## 8.8 증류한 모델 평가하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F16_raschka.webp" width=500>

- 이 노트북에서 증류 학습을 직접 돌리는 대신, 아래의 편의 스크립트를 내려받아 쓸 수 있습니다.

In [ ]:
# from reasoning_from_scratch.ch07 import download_from_github

# 증류 학습 스크립트 distill.py를 내려받습니다
download_from_github(
    "ch08/04_train_with_distillation/distill.py"
)

```bash
uv run distill.py \
--data_path deepseek-r1-math-train.json \
--validation_size 25 \
--epochs 3 \
--lr 1e-5 \
--max_seq_len 2048 \
--use_think_tokens \
--grad_clip 1.0
```

- 위 설정으로 DGX Spark에서 학습하면 GPU 메모리 15.02GB를 쓰고 약 3시간 5분이 걸립니다.
- 직접 돌리고 싶지 않다면, 아래에서 결과 지표 파일을 내려받아 분석합니다.

In [ ]:
# 저자가 미리 학습해 둔 손실 기록(CSV)을 내려받습니다
download_from_github(
    "ch08/03_logs/deepseek-r1-2048_distill_metrics.csv"
)

In [ ]:
import csv
import matplotlib.pyplot as plt

def plot_distill_metrics(csv_path="train_distill_metrics.csv"):
    total_steps, train_losses, val_losses, epoch_bounds = [], [], [], {}

    # CSV에서 스텝·손실·에포크 구간을 읽어 옵니다
    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            step = int(row["total_steps"])
            epoch = int(row["epoch"])
            total_steps.append(step)
            train_losses.append(float(row["train_loss"]))
            val_losses.append(float(row["val_loss"]))
            epoch_bounds.setdefault(epoch, [step, step])[1] = step  # first/last step for each epoch

    fig, ax = plt.subplots(figsize=(7, 4))
    # 학습 손실은 흐리게, 검증 손실은 진하게 그립니다
    ax.plot(total_steps, train_losses, label="train_loss", alpha=0.3)
    ax.plot(total_steps, val_losses, label="val_loss")
    ax.set_xlabel("Total Steps")
    ax.set_ylabel("Loss")
    ax.legend()

    # Epoch axis
    epoch_axis = ax.secondary_xaxis("bottom")
    epoch_axis.spines["bottom"].set_position(("outward", 45))
    epochs = sorted(epoch_bounds)
    epoch_axis.set_xticks(
        [(epoch_bounds[epoch][0] + epoch_bounds[epoch][1]) / 2 
          for epoch in epochs]
    )
    epoch_axis.set_xticklabels(epochs)
    epoch_axis.set_xlabel("Epoch")

    plt.tight_layout()
    plt.show()

# 내려받은 기록으로 손실 곡선을 그립니다
plot_distill_metrics("deepseek-r1-2048_distill_metrics.csv")

- 학습 손실은 샘플 하나로 계산해서 들쭉날쭉합니다.
- 검증 손실처럼 여러 샘플(예: 25개)로 평가할 수도 있습니다. 하지만 그러면 학습 시간이 늘어납니다.
- 우리가 보고 싶은 것은 검증 손실입니다. 이 값이 줄어들기를 바랍니다.
- 검증 손실은 가파르게 줄다가 그 뒤로는 크게 나아지지 않습니다.
- 학습률을 올려 더 세게 학습시킬 수도 있습니다. 하지만 전체적으로 검증 곡선은 좋아 보입니다.

- 7장처럼 3장의 평가 도구로 결과 모델 체크포인트를 평가할 수 있습니다.

In [ ]:
# 3장의 MATH-500 평가 스크립트를 내려받습니다
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)

```bash
uv run evaluate_math500.py \
--dataset_size 500 \
--which_model reasoning \
--max_new_tokens 4096 \
--checkpoint_path run_11/checkpoints/distill/qwen3-0.6B-distill-step05746-epoch1.pth
```

|      | 선생 데이터                          | 에포크 | MATH-500 정확도 | 최종 검증 손실 |
| ---- | ------------------------------------ | ----- | ------------ | -------------- |
| 1    | 기본 모델(3장)                        | -     | 15.2%        | -              |
| 2    | 추론 모델(3장)                        | -     | 48.2%        | -              |
| 3    | DeepSeek R1 증류 데이터               | 1     | 30.6%        | 0.5436         |
| 4    | DeepSeek R1 증류 데이터               | 2     | 32.4%        | 0.5349         |
| 5    | DeepSeek R1 증류 데이터               | 3     | 33.6%        | 0.5343         |
| 6    | Qwen3 235B A22B 증류 데이터           | 1     | 45.0%        | 0.4043         |
| 7    | Qwen3 235B A22B 증류 데이터           | 2     | 43.8%        | 0.3963         |
| 8    | Qwen3 235B A22B 증류 데이터           | 3     | 44.2%        | 0.3948         |

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch08/CH08_F18_raschka.webp" width=500>

&nbsp;
## 8.9 추론 모델의 앞으로의 방향

- 이 절에는 코드가 없습니다.
- 더 많은 참고 자료는 부록 A를 보세요.

&nbsp;
## 8.10 맺음말

- 이 절에는 코드가 없습니다.

- 이 책을 즐겁게 읽으셨기를 바랍니다!
- 본문은 여기서 끝납니다. 하지만 쓸모 있을 만한 “부록” 장들이 더 있습니다. LLM 구조의 세부 내용, 처리량을 높이는 배치 모드 실행, 여러 평가 방법 등을 다룹니다.
- 보너스 자료는 [#bonus-material](https://github.com/rasbt/reasoning-from-scratch?tab=readme-ov-file#bonus-material) 섹션에서 더 볼 수 있습니다.

- 저자는 블로그 https://magazine.sebastianraschka.com 에 AI와 LLM 주제의 글을 꾸준히 씁니다. 도움이 될 수 있습니다.

&nbsp;
## 요약

- 이 절에는 코드가 없습니다.